# 01 - Total p-Variation (TpV) Reconstruction for Sparse-View CT

Run Total p-Variation reconstruction on the fixed sparse-view CT test data.
The solver uses `IPPy.solvers.ChambollePockTpVUnconstrained` and evaluates all sparse-view settings on the same central test slice.

## 1. Environment Setup and Imports

Install dependencies, mount Drive, set paths, import libraries, and configure the execution device.

In [ ]:
!pip install astra-toolbox

from google.colab import drive

drive.mount("/content/drive")

import sys
from pathlib import Path
import json
import numpy as np
import torch
import matplotlib.pyplot as plt

# Paths setup
PROJECT_ROOT = Path("/content/drive/MyDrive/LM_INFORMATICA/COMPUTATIONAL_IMAGING")
PROCESSED_DIR = PROJECT_ROOT / "processed2"
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "tpv"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

sys.path.append(str(PROJECT_ROOT))

from IPPy import operators, solvers, utilities
from IPPy.utilities import normalize

# Set seed for reproducibility
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

# Device configuration (CPU or GPU)
device = utilities.get_device()
print("Device used:", device)
print("Processed data directory:", PROCESSED_DIR)
print("TpV outputs directory:", OUTPUT_DIR)

## 2. Load the Preprocessed Test Data Contract

Load the first test patient from the processed dataset manifest.

In [ ]:
manifest_path = PROCESSED_DIR / "manifest.json"
with manifest_path.open("r", encoding="utf-8") as file:
    manifest = json.load(file)

def load_first_processed_patient(split_name: str) -> dict:
    split_info = manifest["splits"][split_name]
    patient_record = split_info["patients"][0]
    patient_path = PROCESSED_DIR / patient_record["path"]
    payload = torch.load(patient_path, map_location="cpu")
    return {
        "split": split_name,
        "path": patient_path,
        "clean": payload["clean"].to(torch.float32),
        "sinograms": {key: val.to(torch.float32) for key, val in payload["sinograms"].items()},
        "metadata": payload["metadata"],
    }

test_data = load_first_processed_patient("test")

metadata = test_data["metadata"]
print(f"Loaded {test_data['split']} patient ID: {metadata['patient_id']}")
print(f"  clean images shape: {tuple(test_data['clean'].shape)}")
for views_key, sino in test_data["sinograms"].items():
    print(f"  sinogram ({views_key} views) shape: {tuple(sino.shape)}")

## 3. Configure CT Projector and TpV Solver

Create one parallel-beam CT projector and one TpV solver for each view count.

In [ ]:
# Sparse-view settings to test
ANGLE_COUNTS = (180, 90, 60, 45)
IMAGE_SIZE = 256
DETECTOR_SIZE = 256
GEOMETRY = "parallel"
angle_keys = [str(n_views) for n_views in ANGLE_COUNTS]

solver_device = torch.device("cpu")

# Initialize one physical parallel-beam projector and TpV solver for each view setting.
projectors = {
    angle_key: operators.CTProjector(
        img_shape=(IMAGE_SIZE, IMAGE_SIZE),
        angles=np.linspace(0.0, np.pi, int(angle_key)),
        det_size=DETECTOR_SIZE,
        geometry=GEOMETRY,
        force_cpu=True,  # Set to False if CUDA/GPU support is fully configured in ASTRA
    )
    for angle_key in angle_keys
}

tpv_solvers = {
    angle_key: solvers.ChambollePockTpVUnconstrained(projector)
    for angle_key, projector in projectors.items()
}

print("Initialized CT projectors and TpV solvers for views:", ", ".join(angle_keys))
print("TpV solver device:", solver_device)

## 4. Lambda Selection by Discrepancy Principle

Define shared TpV hyperparameters, then estimate $\lambda$ on a training slice by matching the reconstruction residual to the expected noise level.

In [ ]:
# ----------------------- TPV HYPERPARAMETERS ------------------------
lmbda = 0.01          # Fallback/manual regularization parameter
p = 0.35              # Sparsity parameter (strictly between 0.1 and 0.5)
maxiter = 300         # Maximum Chambolle-Pock iterations
tolf = 1e-6           # Residual stopping tolerance
tolx = 1e-6           # Iterate-change stopping tolerance
# --------------------------------------------------------------------

if not (0.1 < p < 0.5):
    raise ValueError(f"Project specifications require 0.1 < p < 0.5. Got p = {p}")

In [ ]:
# -------------------- LAMBDA SELECTION: DISCREPANCY PRINCIPLE --------------------
NOISE_LEVEL = 0.005
TAU_DISCREPANCY = 1.05
LAMBDA_SELECTION_ANGLE_KEY = str(ANGLE_COUNTS[-1])  # Change this if tuning for a different view count
LAMBDA_DISCREPANCY_GRID = np.logspace(-4, 0, 9)
# ---------------------------------------------------------------------------------

lambda_selection_data = load_first_processed_patient("train")

def run_lambda_discrepancy_sweep(split_data: dict, angle_key: str) -> dict:
    clean_images = split_data["clean"]
    sinograms = split_data["sinograms"]
    slice_idx = clean_images.shape[0] // 2

    solver = tpv_solvers[angle_key]
    x_true = clean_images[slice_idx : slice_idx + 1].to(solver_device)
    y_delta = sinograms[angle_key][slice_idx : slice_idx + 1].to(solver_device)

    with torch.no_grad():
        y_clean = solver.K(x_true)
        target_residual = float((TAU_DISCREPANCY * NOISE_LEVEL * torch.norm(y_clean)).detach().cpu())

    sweep_results = []
    for candidate_lmbda in LAMBDA_DISCREPANCY_GRID:
        print(f"Discrepancy sweep | views={angle_key} | lambda={candidate_lmbda:.2e}")
        x_candidate, info = solver(
            y_delta,
            lmbda=float(candidate_lmbda),
            starting_point=None,
            x_true=x_true,
            maxiter=maxiter,
            tolf=tolf,
            tolx=tolx,
            p=p,
            verbose=False,
        )

        with torch.no_grad():
            residual_norm = float(torch.norm(solver.K(x_candidate) - y_delta).detach().cpu())
            relative_error = float(
                (torch.norm((x_candidate - x_true).flatten()) / (torch.norm(x_true.flatten()) + 1e-12)).detach().cpu()
            )

        sweep_results.append(
            {
                "lambda": float(candidate_lmbda),
                "residual_norm": residual_norm,
                "target_residual": target_residual,
                "discrepancy_gap": abs(residual_norm - target_residual),
                "relative_error": relative_error,
                "psnr": float(info["PSNR"][-1, 0]),
                "ssim": float(info["SSIM"][-1, 0]),
                "iterations": int(info["iterations"]),
            }
        )

    best_by_discrepancy = min(sweep_results, key=lambda item: item["discrepancy_gap"])
    return {
        "angle_key": angle_key,
        "slice_idx": slice_idx,
        "target_residual": target_residual,
        "lambda": best_by_discrepancy["lambda"],
        "best": best_by_discrepancy,
        "results": sweep_results,
    }

discrepancy_selection = run_lambda_discrepancy_sweep(lambda_selection_data, LAMBDA_SELECTION_ANGLE_KEY)
lmbda = discrepancy_selection["lambda"]

lambda_selection_summary_path = OUTPUT_DIR / f"tpv_lambda_discrepancy_{LAMBDA_SELECTION_ANGLE_KEY}.json"
with lambda_selection_summary_path.open("w", encoding="utf-8") as file:
    json.dump(discrepancy_selection, file, indent=2)

print("=" * 72)
print(f"Discrepancy target residual: {discrepancy_selection['target_residual']:.6e}")
print(f"Selected lambda by discrepancy principle: {lmbda:.6e}")
print("Saved lambda selection summary:", lambda_selection_summary_path)
print("=" * 72)

## 5. Heuristic Lambda Sweep

Plot training relative error across the tested $\lambda$ values.

In [ ]:
# -------------------- LAMBDA SELECTION: HEURISTIC RELATIVE ERROR CURVE --------------------
heuristic_lambda_results = discrepancy_selection["results"]
best_by_relative_error = min(heuristic_lambda_results, key=lambda item: item["relative_error"])
heuristic_lmbda = best_by_relative_error["lambda"]

lambda_values = np.array([item["lambda"] for item in heuristic_lambda_results])
relative_errors = np.array([item["relative_error"] for item in heuristic_lambda_results])

fig, ax = plt.subplots(1, 1, figsize=(7, 5), constrained_layout=True)
fig.suptitle(f"TpV Lambda Selection - {LAMBDA_SELECTION_ANGLE_KEY} views", fontsize=14)

ax.plot(lambda_values, relative_errors, marker="o")
ax.axvline(heuristic_lmbda, color="tab:red", linestyle="--", label=f"heuristic lambda={heuristic_lmbda:.2e}")
ax.set_xscale("log")
ax.set_title("Relative Error vs Lambda")
ax.set_xlabel("lambda")
ax.set_ylabel(r"$\|x_\lambda - x\|_2 / \|x\|_2$")
ax.grid(True, alpha=0.3)
ax.legend()

heuristic_fig_path = OUTPUT_DIR / f"tpv_lambda_heuristic_{LAMBDA_SELECTION_ANGLE_KEY}.png"
fig.savefig(heuristic_fig_path, dpi=150)
plt.show()
plt.close(fig)

print("Best lambda by relative-error heuristic:", heuristic_lmbda)
print("Best lambda by discrepancy principle:", lmbda)
print("Saved heuristic lambda plot:", heuristic_fig_path)

## 6. Multi-View Final Test Evaluation

Run TpV reconstruction on the shared test slice for every sparse-view setting.

In [ ]:
# -------------------- FINAL TEST HYPERPARAMETERS --------------------
# lmbda, p, maxiter, tolf, and tolx are defined once above.
# lmbda is overwritten by the discrepancy-principle cell if that cell is executed.
# Set lmbda manually here only if you want to override the selected/fallback value.
# -------------------------------------------------------------------------------

if not (0.1 < p < 0.5):
    raise ValueError(f"Project specifications require 0.1 < p < 0.5. Got p = {p}")

def move_info_to_cpu(info: dict) -> dict:
    return {
        key: value.detach().cpu() if isinstance(value, torch.Tensor) else value
        for key, value in info.items()
    }

def run_tpv_reconstruction(split_data: dict, angle_key: str, solver) -> dict:
    clean_images = split_data["clean"]
    sinograms = split_data["sinograms"]
    split_name = split_data["split"]

    # Select the fixed central slice for consistency across all view settings.
    slice_idx = clean_images.shape[0] // 2
    print(f"\nRunning {split_name} reconstruction for {angle_key} views on central slice index: {slice_idx}")

    x_true = clean_images[slice_idx : slice_idx + 1].to(solver_device)
    y_delta = sinograms[angle_key][slice_idx : slice_idx + 1].to(solver_device)

    x_sol, info = solver(
        y_delta,
        lmbda=lmbda,
        starting_point=None,  # Standard zero-image initialization
        x_true=x_true,
        maxiter=maxiter,
        tolf=tolf,
        tolx=tolx,
        p=p,
        verbose=True,
    )

    info = move_info_to_cpu(info)
    x_sol_cpu = x_sol.detach().cpu()
    x_true_cpu = x_true.detach().cpu()
    x_sol_norm = normalize(x_sol_cpu).clamp(0.0, 1.0)

    psnr_val = float(info["PSNR"][-1, 0])
    ssim_val = float(info["SSIM"][-1, 0])

    print(f"{split_name} PSNR: {psnr_val:.4f} dB")
    print(f"{split_name} SSIM: {ssim_val:.4f}")

    return {
        "split": split_name,
        "angle_key": angle_key,
        "slice_idx": slice_idx,
        "x_true_cpu": x_true_cpu,
        "x_sol_cpu": x_sol_cpu,
        "x_sol_norm": x_sol_norm,
        "info": info,
        "iterations": info["iterations"],
        "psnr": psnr_val,
        "ssim": ssim_val,
    }

tpv_results = {
    angle_key: run_tpv_reconstruction(test_data, angle_key, tpv_solvers[angle_key])
    for angle_key in angle_keys
}

# Keep a final active result for compatibility with downstream visualization/reporting cells.
test_result = tpv_results[str(ANGLE_COUNTS[-1])]
active_result = test_result
active_split = active_result["split"]
angle_key = active_result["angle_key"]
n_views = int(angle_key)
slice_idx = active_result["slice_idx"]
x_true_cpu = active_result["x_true_cpu"]
x_sol_cpu = active_result["x_sol_cpu"]
x_sol_norm = active_result["x_sol_norm"]
psnr_val = active_result["psnr"]
ssim_val = active_result["ssim"]

## 7. Visual Inspection

Save reconstruction panels and convergence plots for each view count.

In [ ]:
def info_series(info: dict, key: str) -> np.ndarray:
    values = info[key].detach().cpu().numpy() if isinstance(info[key], torch.Tensor) else np.asarray(info[key])
    return np.atleast_1d(np.squeeze(values)).astype(float)

for angle_key in angle_keys:
    result = tpv_results[angle_key]
    current_n_views = int(angle_key)
    current_slice_idx = result["slice_idx"]
    current_info = result["info"]

    gt_np = result["x_true_cpu"][0, 0].numpy()
    noisy_sino_np = test_data["sinograms"][angle_key][current_slice_idx].squeeze().numpy()
    recon_np = result["x_sol_norm"][0, 0].numpy()

    fig, axes = plt.subplots(1, 3, figsize=(15, 5), constrained_layout=True)
    fig.suptitle(
        f"TpV Test Reconstruction Panel - {current_n_views} views - Slice {current_slice_idx}\n"
        f"(lmbda={lmbda}, p={p}, iterations={result['iterations']})",
        fontsize=14,
    )

    axes[0].imshow(gt_np, cmap="gray", vmin=0.0, vmax=1.0)
    axes[0].set_title("Ground Truth")
    axes[0].axis("off")

    axes[1].imshow(noisy_sino_np.T, cmap="gray")
    axes[1].set_title("Noisy Sinogram")
    axes[1].axis("off")

    axes[2].imshow(recon_np, cmap="gray", vmin=0.0, vmax=1.0)
    axes[2].set_title(f"TpV Reconstruction\nPSNR: {result['psnr']:.2f} dB | SSIM: {result['ssim']:.4f}")
    axes[2].axis("off")
    output_fig_path = OUTPUT_DIR / f"tpv_test_reconstruction_panel_{angle_key}.png"
    fig.savefig(output_fig_path, dpi=150)
    plt.show()
    plt.close(fig)
    print("Saved TpV reconstruction panel:", output_fig_path)

    iteration_axis = np.arange(1, len(info_series(current_info, "obj")) + 1)
    fig, axes = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)
    fig.suptitle(f"TpV Convergence - {current_n_views} views - Slice {current_slice_idx}", fontsize=14)

    axes[0].plot(iteration_axis, info_series(current_info, "PSNR"))
    axes[0].set_title("PSNR")
    axes[0].set_xlabel("Iteration")
    axes[0].grid(True, alpha=0.3)

    axes[1].plot(iteration_axis, info_series(current_info, "SSIM"))
    axes[1].set_title("SSIM")
    axes[1].set_xlabel("Iteration")
    axes[1].grid(True, alpha=0.3)

    for ax in axes:
        if len(iteration_axis) == 1:
            ax.set_xlim(0.5, 1.5)
        else:
            ax.set_xlim(1, len(iteration_axis))

    convergence_fig_path = OUTPUT_DIR / f"tpv_convergence_{angle_key}.png"
    fig.savefig(convergence_fig_path, dpi=150)
    plt.show()
    plt.close(fig)
    print("Saved TpV convergence panel:", convergence_fig_path)

## 8. Quantitative Results

Print the final TpV metrics and reconstruction parameters.

In [ ]:
print("="*140)
print(
    f"{ 'VIEWS':<8} | { 'METHOD':<12} | { 'LAMBDA':<12} | { 'P':<6} | "
    f"{ 'MAXITER':<8} | { 'TOLF':<10} | { 'TOLX':<10} | { 'ITERATIONS':<10} | { 'PSNR (dB)':<12} | { 'SSIM':<8}"
)
print("="*140)
for angle_key in angle_keys:
    result = tpv_results[angle_key]
    print(
        f"{ angle_key:<8} | { 'TpV test':<12} | { lmbda:<12.4e} | { p:<6.2f} | "
        f"{ maxiter:<8} | { tolf:<10.1e} | { tolx:<10.1e} | { result['iterations']:<10} | { result['psnr']:<12.4f} | { result['ssim']:<8.4f}"
    )
print("="*140)